# Import Libraries

In [44]:
import torch
from torch.utils.data import Subset, Dataset, DataLoader
from tqdm import tqdm
import multiprocessing as mp

from models.gpt import GPT
from dataset.dataset import OthelloDataset
from game.othello import GameBoard, Piece

# Load  a target model

In [3]:
def load_checkpoint(model, checkpoint):
    checkpoint = torch.load("../checkpoints/" + checkpoint)
    model.load_state_dict(checkpoint)

model = GPT()
load_checkpoint(model, "gpt_batch_0_loss_2.0953_pass_rate_96.1313.pt")

# Create dataset

In [43]:
NUM_OF_GAMES_FOR_PROBE_TRAINING = 21_000
NUM_OF_GAMES_USED_TO_EVALUATE_GPT = 1000
MAX_GAME_LENGTH = 60
BOARD_SIZE = 64
PADDING_TOKEN = 0

test_dataset = OthelloDataset(train=False, path="../dataset/test_dataset.pt")
probe_dataset = Subset(test_dataset, range(NUM_OF_GAMES_USED_TO_EVALUATE_GPT, NUM_OF_GAMES_FOR_PROBE_TRAINING + NUM_OF_GAMES_USED_TO_EVALUATE_GPT))

def convert_token_to_position(token):
    flat_index = token - 1

    if flat_index >= 33:
        flat_index += 4

    elif flat_index >= 27:
        flat_index += 2

    row = flat_index // 8
    col = flat_index % 8

    return GameBoard.index_to_position((row, col))


# Calculate total number of sequences
num_of_sequences = 0
temp = torch.empty((NUM_OF_GAMES_FOR_PROBE_TRAINING, MAX_GAME_LENGTH))
for i in range(NUM_OF_GAMES_FOR_PROBE_TRAINING):
    temp[i] = probe_dataset[i][0]
num_of_sequences = temp.ne(0).sum().item()

# Create X and Y
X = torch.full((num_of_sequences, MAX_GAME_LENGTH), PADDING_TOKEN)
Y = torch.full((num_of_sequences, 64, 3), 0)
idx = 0
for i in range(NUM_OF_GAMES_FOR_PROBE_TRAINING):
    x = probe_dataset[i][0]
    for j in range(MAX_GAME_LENGTH):
        if x[j] == PADDING_TOKEN:
            break

        X[idx, :j + 1] = x[:j + 1]
        idx += 1

def process_sequence(args):
    i, x_seq = args
    x_filtered = x_seq[x_seq.ne(PADDING_TOKEN)]
    list_of_positions = [convert_token_to_position(token.item()) for token in x_filtered]

    game_board = GameBoard()
    current_player = Piece.BLACK

    for position in list_of_positions:
        game_board.add_piece(current_player, position)

        current_player = Piece.WHITE if current_player == Piece.BLACK else Piece.BLACK
        if not game_board.get_legal_moves(current_player):
            opponent = Piece.WHITE if current_player == Piece.BLACK else Piece.BLACK
            if game_board.get_legal_moves(opponent):
                current_player = opponent

    game_board = game_board.get_board().flatten()
    y = torch.zeros((BOARD_SIZE, 3))
    opponent = Piece.WHITE if current_player == Piece.BLACK else Piece.BLACK

    mine_mask = (game_board == current_player)
    yours_mask = (game_board == opponent)
    empty_mask = ~(mine_mask | yours_mask)

    y[mine_mask, 0] = 1.0
    y[yours_mask, 1] = 1.0
    y[empty_mask, 2] = 1.0

    return i, y

if __name__ == '__main__':
    tasks = [(i, X[i]) for i in range(num_of_sequences)]
    with mp.Pool(processes=mp.cpu_count()) as pool:
        for i, y in tqdm(pool.imap_unordered(process_sequence, tasks, chunksize=250),
                         total=num_of_sequences,
                         desc="Generating Y labels"):
            Y[i] = y

    torch.save(X, "sequences.pt")
    torch.save(Y, "game_boards.pt")

Generating Y labels (Pool): 100%|██████████| 1259628/1259628 [24:54<00:00, 842.69it/s] 


# Load dataset

In [ ]:
class ProbeDataset(Dataset):
    def __init__(self, train = True):
        X =

    def __len__(self):
        pass

    def __getitem__(self, item):
        pass